In [ ]:
# @title PyKinshipID: DVI Screening Tool  - Automated STR Profile Analysis for Disaster Victim Identification.

# @markdown ---

# ── static launch banner ─────────────────────────────────────────────────
from IPython.display import display as _display, HTML as _HTML
_display(_HTML("""
<div style='background:#1a3a4a;color:#fff;padding:32px 40px;
            border-radius:8px;text-align:center;font-family:Arial,sans-serif'>
  <div style='font-size:28px;font-weight:bold;margin-bottom:10px'>
    &#x1F9EC; PyKinshipID: DVI Screening Tool</div>
  <div style='font-size:14px;color:#d0e8f0;margin-bottom:20px'>
    Automated STR Profile Analysis for Disaster Victim Identification</div>
  <div style='background:#2e6b5e;border-radius:6px;padding:14px 28px;
              display:inline-block;font-size:15px;font-weight:bold'>
    &#9654; Click the Play button above to launch the tool</div>
  <div style='font-size:11px;color:#a0c4cc;margin-top:14px'>
    No programming knowledge required &nbsp;&#183;&nbsp;
    All code is hidden &nbsp;&#183;&nbsp; For research and screening use only</div>
</div>
"""))
# ── end banner ────────────────────────────────────────────────────────────


# ── lightweight imports at cell load ─────────────────────────────────────────
import io, os, base64, warnings, time, hashlib, itertools
from collections import defaultdict
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
warnings.filterwarnings("ignore", category=FutureWarning)

# ── shared state ──────────────────────────────────────────────────────────────
_state = {}
_state["cancel"] = False

VERSION   = "PyKinshipID"
APP_TITLE = "PyKinshipID: DVI Screening Tool"

STR_LOCI = [
    "D8S1179","D21S11","D7S820","CSF1PO","D3S1358","TH01",
    "D13S317","D16S539","D2S1338","D19S433","vWA","TPOX",
    "D18S51","D5S818","FGA"
]
AMELOGENIN    = "Amelogenin"
SAMPLE_ID_COL = "SampleID"

AMEL_ALIASES = {"amel","amelogenin","sex","gender_marker"}
LOCUS_ALIASES = {
    "d8s1179":"D8S1179","d21s11":"D21S11","d21":"D21S11",
    "d7s820":"D7S820","csf1po":"CSF1PO","csf":"CSF1PO",
    "d3s1358":"D3S1358","th01":"TH01","th0.1":"TH01",
    "d13s317":"D13S317","d16s539":"D16S539","d2s1338":"D2S1338",
    "d19s433":"D19S433","vwa":"vWA","tpox":"TPOX",
    "d18s51":"D18S51","d5s818":"D5S818","fga":"FGA",
    "sampleid":"SampleID","sample_id":"SampleID",
    "sample id":"SampleID","sample":"SampleID","id":"SampleID",
}

# ── kit signature database ────────────────────────────────────────────────────
# Each entry: (display_name, manufacturer, unique_markers, absent_core_loci, notes)
KIT_SIGNATURES = [
    {
        "name": "GlobalFiler",
        "manufacturer": "Thermo Fisher Scientific",
        "markers": {"SE33","DYS391"},
        "absent_core": [],
        "extra_markers": {"D1S1656","D2S441","D10S1248","D22S1045","D12S391","SE33","DYS391"},
        "notes": "All 15 core loci present. Extra loci ignored by PyKinshipID."
    },
    {
        "name": "PowerPlex Fusion",
        "manufacturer": "Promega",
        "markers": {"Penta_D","Penta_E"},
        "absent_core": [],
        "extra_markers": {"D1S1656","D2S441","D10S1248","D22S1045","D12S391","SE33","Penta_D","Penta_E"},
        "notes": "All 15 core loci present. Penta D/E detected but not used — outside the 15 core IDENTIFILER PLUS loci."
    },
    {
        "name": "PowerPlex 16",
        "manufacturer": "Promega",
        "markers": {"Penta_D","Penta_E"},
        "absent_core": ["D2S1338","D19S433"],
        "extra_markers": {"Penta_D","Penta_E"},
        "notes": (
            "⚠ D2S1338 and D19S433 are absent from this kit. "
            "These are highly discriminating loci (DP ~0.981 and ~0.952 respectively). "
            "Kinship conclusions from this sheet have reduced statistical support. "
            "Confirmation with a full 15-locus panel is strongly recommended."
        )
    },
    {
        "name": "Investigator 24plex QS",
        "manufacturer": "QIAGEN / Verasyte",
        "markers": {"SE33"},
        "absent_core": [],
        "extra_markers": {"D1S1656","D2S441","D10S1248","D22S1045","D12S391","SE33"},
        "notes": "All 15 core loci present. Extra loci ignored by PyKinshipID."
    },
    {
        "name": "NGM SElect",
        "manufacturer": "Thermo Fisher Scientific",
        "markers": {"SE33"},
        "absent_core": [],
        "extra_markers": {"D1S1656","D2S441","D10S1248","D22S1045","SE33"},
        "notes": "All 15 core loci present. Extra loci ignored by PyKinshipID."
    },
    {
        "name": "IDENTIFILER PLUS",
        "manufacturer": "Thermo Fisher Scientific",
        "markers": set(),
        "absent_core": [],
        "extra_markers": set(),
        "notes": "Standard 15-locus panel — fully compatible."
    },
]

# Penta locus column name variants
PENTA_ALIASES = {
    "penta_d","pentad","penta d","penta-d",
    "penta_e","pentae","penta e","penta-e",
}

def detect_kit(col_names):
    """
    Identify the STR kit from column names present in a sheet.
    Returns dict with keys: name, manufacturer, notes, absent_core,
    penta_present, se33_present, dys391_present.
    """
    cols_lower = {str(c).strip().lower() for c in col_names}
    cols_orig  = {str(c).strip() for c in col_names}

    has_se33    = any(c in cols_lower for c in ("se33",))
    has_dys391  = any(c in cols_lower for c in ("dys391",))
    has_penta   = any(c in cols_lower for c in PENTA_ALIASES)
    has_d2s1338 = "d2s1338" in cols_lower
    has_d19s433 = "d19s433" in cols_lower

    # Priority-ordered matching
    if has_se33 and has_dys391 and not has_penta:
        kit = next(k for k in KIT_SIGNATURES if k["name"] == "GlobalFiler")
    elif has_penta and not has_d2s1338 and not has_d19s433:
        kit = next(k for k in KIT_SIGNATURES if k["name"] == "PowerPlex 16")
    elif has_penta and has_d2s1338:
        kit = next(k for k in KIT_SIGNATURES if k["name"] == "PowerPlex Fusion")
    elif has_se33 and not has_dys391 and not has_penta:
        # Could be Investigator 24plex or NGM SElect — distinguish by D22S1045
        has_d22 = "d22s1045" in cols_lower
        if has_d22:
            kit = next(k for k in KIT_SIGNATURES if k["name"] == "Investigator 24plex QS")
        else:
            kit = next(k for k in KIT_SIGNATURES if k["name"] == "NGM SElect")
    else:
        kit = next(k for k in KIT_SIGNATURES if k["name"] == "IDENTIFILER PLUS")

    return {
        "name":           kit["name"],
        "manufacturer":   kit["manufacturer"],
        "notes":          kit["notes"],
        "absent_core":    kit["absent_core"],
        "penta_present":  has_penta,
        "se33_present":   has_se33,
        "dys391_present": has_dys391,
    }

# ── forensic palette ──────────────────────────────────────────────────────────
C = dict(
    navy   ="#1a3a4a", teal   ="#2e6b5e", green  ="#2d5a3d",
    slate  ="#3a5068", light  ="#e8eef2", pale   ="#f4f7f9",
    border ="#b0bec5", text   ="#1a2a35", muted  ="#546e7a",
    white  ="#ffffff", red    ="#7b1a1a", amber  ="#7a4a00",
    male   ="#1a3a6a", female ="#6a1a2e", unknown="#3a3a3a",
)

DISCLAIMER = (
    "<b>⚠ Screening Disclaimer:</b> PyKinshipID is a preliminary screening tool only. "
    "Results are based on Mendelian mismatch counting and do <b>not</b> incorporate "
    "likelihood ratios or population allele frequencies. All positive kinship findings "
    "must be confirmed by a qualified forensic geneticist using validated LR-based "
    "software before any formal identification is made."
)

# ── static UI strings ─────────────────────────────────────────────────────────
_FORMAT_HTML = f"""
<style>
  details.pykin-accordion summary {{
    list-style: none;
    cursor: pointer;
    user-select: none;
  }}
  details.pykin-accordion summary::-webkit-details-marker {{ display: none; }}
  details.pykin-accordion summary .pykin-chevron {{
    display: inline-block;
    margin-right: 6px;
    transition: transform 0.2s ease;
    color: {C['slate']};
    font-size: 13px;
  }}
  details.pykin-accordion[open] summary .pykin-chevron {{
    transform: rotate(90deg);
  }}
</style>
<details class='pykin-accordion' style='border:1px solid {C['slate']};border-radius:4px;
            background:{C['pale']};margin-bottom:10px;overflow:hidden'>
  <summary style='padding:12px 18px;display:flex;align-items:center;
                  background:{C['pale']};border-radius:4px'>
    <span class='pykin-chevron'>&#9654;</span>
    <b style='color:{C['navy']}'>📋 Required Data Format</b>
    <span style='font-size:11px;color:{C['muted']};margin-left:8px'>
      Click to view — prepare your file to match this structure before uploading</span>
  </summary>
  <div style='padding:0 18px 14px 18px;border-top:1px solid {C['border']}'>
  <table style='border-collapse:collapse;font-size:12px;width:100%;margin-top:12px'>
    <tr style='background:{C['navy']};color:#fff'>
      <th style='padding:5px 8px;text-align:left'>SampleID</th>
      <th style='padding:5px 8px'>Amelogenin</th>
      <th style='padding:5px 8px'>D8S1179</th><th style='padding:5px 8px'>D21S11</th>
      <th style='padding:5px 8px'>D7S820</th><th style='padding:5px 8px'>CSF1PO</th>
      <th style='padding:5px 8px'>…</th><th style='padding:5px 8px'>FGA</th>
    </tr>
    <tr style='background:#fff'>
      <td style='padding:5px 8px;border:1px solid {C['border']};color:{C['text']}'>Victim_001</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>X,Y</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>13,14</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>26,28</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>10,11</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>11,13</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>…</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>19,23</td>
    </tr>
    <tr style='background:{C['pale']}'>
      <td style='padding:5px 8px;border:1px solid {C['border']};color:{C['text']}'>Ref_Mother_01</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>X,X</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>13,15</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>25,38</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>11,12</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>7,7</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>…</td>
      <td style='padding:5px 8px;border:1px solid {C['border']};text-align:center;color:{C['text']}'>34,37</td>
    </tr>
  </table>
  <div style='font-size:11px;color:{C['text']};margin-top:10px;line-height:1.9'>
    • Alleles comma-separated (e.g. <code style='background:#e8eef2;padding:1px 4px;border-radius:3px;color:{C['navy']}'>13,14</code>) &nbsp;·&nbsp;
      Microvariants supported (e.g. <code style='background:#e8eef2;padding:1px 4px;border-radius:3px;color:{C['navy']}'>9.3</code>) &nbsp;·&nbsp;
      Leave blank for failed loci — do <b>not</b> use 0,0 &nbsp;·&nbsp;
      SampleID must be unique per profile<br>
    • <b>Multi-sheet Excel:</b> Profiles from different STR kits may be placed on
      separate sheets in the same Excel file — the 15 core IDENTIFILER PLUS loci
      are extracted automatically from each sheet and merged before analysis.
      Father, Mother and Child may each be on different sheets.<br>
    • <b>Kit compatibility:</b> IDENTIFILER PLUS, GlobalFiler, Investigator 24plex QS,
      PowerPlex Fusion, PowerPlex 16, NGM SElect — kits are auto-detected per sheet.
      Extra loci beyond the core 15 are ignored. PowerPlex 16 is missing D2S1338 and
      D19S433 — results from this kit have reduced statistical support.<br>
    • <b>Column aliases:</b> AMEL→Amelogenin, CSF→CSF1PO, D21→D21S11,
      VWA→vWA (case-insensitive).
  </div>
  </div>
</details>"""

_SAMPLE_HTML = f"""
<details class='pykin-accordion' style='border:1px solid {C['green']};border-radius:4px;
            background:#f0f7f0;margin-bottom:10px;overflow:hidden'>
  <summary style='padding:12px 18px;display:flex;align-items:center;
                  background:#f0f7f0;border-radius:4px'>
    <span class='pykin-chevron' style='color:{C['green']}'>&#9654;</span>
    <b style='color:{C['green']}'>📦 Sample Dataset — Built-in Demo Data</b>
    <span style='font-size:11px;color:{C['muted']};margin-left:8px'>
      Click to view — demonstrates all five classification categories</span>
  </summary>
  <div style='padding:0 18px 14px 18px;border-top:1px solid {C['green']}22'>
  <table style='border-collapse:collapse;font-size:12px;width:100%;margin-top:12px'>
    <tr style='background:{C['green']};color:#fff'>
      <th style='padding:5px 10px;text-align:left'>Category</th>
      <th style='padding:5px 10px;text-align:right'>Count</th>
      <th style='padding:5px 10px;text-align:left'>Profile IDs</th>
    </tr>
    <tr style='background:#fff'>
      <td style='padding:5px 10px;border:1px solid {C['border']};color:{C['text']}'>Trios</td>
      <td style='padding:5px 10px;border:1px solid {C['border']};text-align:right;color:{C['text']}'>4</td>
      <td style='padding:5px 10px;border:1px solid {C['border']};font-family:monospace;font-size:11px;color:{C['text']}'>
        TRIO_1…TRIO_4 (Father / Mother / Child each)</td></tr>
    <tr style='background:{C['pale']}'>
      <td style='padding:5px 10px;border:1px solid {C['border']};color:{C['text']}'>Duos</td>
      <td style='padding:5px 10px;border:1px solid {C['border']};text-align:right;color:{C['text']}'>2</td>
      <td style='padding:5px 10px;border:1px solid {C['border']};font-family:monospace;font-size:11px;color:{C['text']}'>
        DUO_1_ProfileA/B, DUO_2_ProfileA/B</td></tr>
    <tr style='background:#fff'>
      <td style='padding:5px 10px;border:1px solid {C['border']};color:{C['text']}'>Duplicates</td>
      <td style='padding:5px 10px;border:1px solid {C['border']};text-align:right;color:{C['text']}'>1 set</td>
      <td style='padding:5px 10px;border:1px solid {C['border']};font-family:monospace;font-size:11px;color:{C['text']}'>
        DUP_Sample_A &amp; DUP_Sample_B</td></tr>
    <tr style='background:{C['pale']}'>
      <td style='padding:5px 10px;border:1px solid {C['border']};color:{C['text']}'>Mixture</td>
      <td style='padding:5px 10px;border:1px solid {C['border']};text-align:right;color:{C['text']}'>1</td>
      <td style='padding:5px 10px;border:1px solid {C['border']};font-family:monospace;font-size:11px;color:{C['text']}'>
        MIX_001 — 3 alleles at D8S1179</td></tr>
    <tr style='background:#fff'>
      <td style='padding:5px 10px;border:1px solid {C['border']};font-weight:bold;color:{C['text']}'>Total</td>
      <td style='padding:5px 10px;border:1px solid {C['border']};text-align:right;font-weight:bold;color:{C['text']}'>19</td>
      <td style='padding:5px 10px;border:1px solid {C['border']};font-size:11px;color:{C['muted']}'>
        15 autosomal STR loci + Amelogenin &nbsp;·&nbsp; TRIO_3 includes TH01 microvariant 9.3</td></tr>
  </table>
  </div>
</details>"""

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — ANALYSIS FUNCTIONS  (unchanged from v5)
# ─────────────────────────────────────────────────────────────────────────────

def parse_alleles(raw):
    s = str(raw).strip()
    if s in ("","nan","none","NaN","None","NONE"): return ()
    parts = [p.strip() for p in s.replace(";",",").split(",") if p.strip()]
    if not parts: return ()
    parts = [p.upper() if p.upper() in ("X","Y") else p for p in parts]
    if len(parts) == 1:
        p = parts[0]
        if p == "X": return ("X","X")
        try:    return (str(float(p)), str(float(p)))
        except: return (p, p)
    def skey(a):
        if a in ("X","Y"): return (1, a)
        try:    return (0, float(a))
        except: return (0, a)
    return tuple(sorted(parts, key=skey))

def infer_gender(amel):
    if amel == ("X","X"): return "Female"
    if amel == ("X","Y"): return "Male"
    return "Uncertain"

def profile_fingerprint(pid, profiles, loci):
    key = str(tuple(profiles[pid].get(l,()) for l in loci))
    return hashlib.md5(key.encode()).hexdigest()

def check_mendelian(ca, pa):
    if not ca or not pa: return False
    return bool(set(ca) & set(pa))

def check_duo(child_id, parent_id, profiles, loci, max_mm):
    mm, mm_loci = 0, []
    c, p = profiles[child_id], profiles[parent_id]
    for locus in loci:
        ca, pa = c.get(locus,()), p.get(locus,())
        if not ca or not pa: continue
        if not check_mendelian(ca, pa):
            mm += 1; mm_loci.append(locus)
        if mm > max_mm: return False, mm, mm_loci
    return mm <= max_mm, mm, mm_loci

def check_trio(child_id, mother_id, father_id, profiles, loci, max_mm):
    mm, mm_loci = 0, []
    c, m, f = profiles[child_id], profiles[mother_id], profiles[father_id]
    for locus in loci:
        ca, ma, fa = c.get(locus,()), m.get(locus,()), f.get(locus,())
        if not ca or not ma or not fa: continue
        possible = {tuple(sorted((mv,fv))) for mv in set(ma) for fv in set(fa)}
        if tuple(sorted(ca)) not in possible:
            mm += 1; mm_loci.append(locus)
        if mm > max_mm: return False, mm, mm_loci
    return mm <= max_mm, mm, mm_loci

def estimate_time(n):
    secs = (n * n / 102000) * 1.2
    if secs < 60: return f"~{max(1,int(secs))} seconds"
    return f"~{secs/60:.1f} minutes"

def normalise_columns(df):
    renames, col_warnings = {}, []
    for col in df.columns:
        cl = str(col).strip().lower()
        if cl in ("sampleid","sample_id","sample id","sample","id"):
            if str(col) != SAMPLE_ID_COL: renames[col] = SAMPLE_ID_COL
        elif cl in AMEL_ALIASES:
            if str(col) != AMELOGENIN:
                renames[col] = AMELOGENIN
                if cl != "amelogenin":
                    col_warnings.append(
                        f"Column '<b>{col}</b>' mapped to Amelogenin.")
        else:
            mapped = LOCUS_ALIASES.get(cl)
            if mapped and str(col) != mapped:
                renames[col] = mapped
    rename_strs = [f"'{k}'→'{v}'" for k,v in renames.items()]
    if renames: df = df.rename(columns=renames)
    return df, rename_strs, col_warnings

def load_excel_sheets(content):
    """
    Load all sheets, detect kit per sheet, merge profiles.
    Returns (merged_df, sheet_summary_list, conflict_warnings).
    """
    import pandas as pd
    xl = pd.ExcelFile(io.BytesIO(content))
    sheet_names   = xl.sheet_names
    sheet_summary = []
    all_frames    = []
    conflict_warnings = []

    seen_profiles = {}

    for sname in sheet_names:
        try:
            df_s = xl.parse(sname)
        except Exception as e:
            sheet_summary.append(dict(
                sheet_name=sname, n_profiles=0, loci_count=0,
                loci_found=[], kit="Unknown", kit_manufacturer="",
                kit_notes="", absent_core=[], status=f"❌ Error: {e}"
            ))
            continue

        if df_s.empty:
            sheet_summary.append(dict(
                sheet_name=sname, n_profiles=0, loci_count=0,
                loci_found=[], kit="Unknown", kit_manufacturer="",
                kit_notes="", absent_core=[], status="⚠ Empty sheet"
            ))
            continue

        # Kit detection BEFORE column normalisation
        kit_info = detect_kit(df_s.columns.tolist())

        df_s, _, _ = normalise_columns(df_s)
        loci_found = [l for l in STR_LOCI if l in df_s.columns]

        if not loci_found or SAMPLE_ID_COL not in df_s.columns:
            sheet_summary.append(dict(
                sheet_name=sname, n_profiles=0, loci_count=0,
                loci_found=[], kit=kit_info["name"],
                kit_manufacturer=kit_info["manufacturer"],
                kit_notes=kit_info["notes"],
                absent_core=kit_info["absent_core"],
                status="⚠ No recognised STR loci — skipped"
            ))
            continue

        df_s["_sheet"] = sname
        df_s["_kit"]   = kit_info["name"]
        n = len(df_s)

        # Kit-specific status
        status_parts = [f"✅ {n} profiles · {len(loci_found)}/{len(STR_LOCI)} loci"]
        if kit_info["absent_core"]:
            status_parts.append(
                f"⚠ Missing core loci: {', '.join(kit_info['absent_core'])}"
            )
        if kit_info["penta_present"]:
            status_parts.append("ℹ Penta D/E present but not used")

        sheet_summary.append(dict(
            sheet_name=sname, n_profiles=n,
            loci_count=len(loci_found), loci_found=loci_found,
            kit=kit_info["name"],
            kit_manufacturer=kit_info["manufacturer"],
            kit_notes=kit_info["notes"],
            absent_core=kit_info["absent_core"],
            status=" &nbsp;·&nbsp; ".join(status_parts)
        ))
        all_frames.append(df_s)

        # Conflict detection across sheets
        loci_present = loci_found
        for _, row in df_s.iterrows():
            sid = str(row.get(SAMPLE_ID_COL,"")).strip()
            if not sid or sid.lower() in ("nan","none"): continue
            fp = str(tuple(
                parse_alleles(str(row.get(l,""))) for l in loci_present))
            if sid in seen_profiles:
                prev_fp, prev_sheet = seen_profiles[sid]
                if prev_fp != fp:
                    conflict_warnings.append(
                        f"<b>Conflicting data:</b> SampleID '<b>{sid}</b>' "
                        f"appears in sheets '<b>{prev_sheet}</b>' and "
                        f"'<b>{sname}</b>' with different allele values — "
                        f"last occurrence will be used. Verify data integrity."
                    )
            seen_profiles[sid] = (fp, sname)

    if not all_frames:
        import pandas as pd
        return pd.DataFrame(), sheet_summary, conflict_warnings

    import pandas as pd
    merged = pd.concat(all_frames, ignore_index=True, sort=False)
    return merged, sheet_summary, conflict_warnings

def run_analysis(df, max_mm, progress_cb, cancel_check):
    import pandas as pd
    import numpy as np

    t0 = time.time()
    results = dict(
        warnings=[], mixtures={}, duplicates=[],
        trios=[], duos=[], individuals=[],
        profiles={}, loci_used=[], cancelled=False,
        column_renames=[], col_warnings=[]
    )

    progress_cb("Data cleaning & column normalisation", 2, time.time()-t0)
    df, rename_strs, col_warnings = normalise_columns(df)
    results["column_renames"] = rename_strs
    results["col_warnings"]   = col_warnings

    if SAMPLE_ID_COL not in df.columns:
        raise ValueError(f"Column '{SAMPLE_ID_COL}' not found.")
    loci_present = [l for l in STR_LOCI if l in df.columns]
    if not loci_present:
        raise ValueError("No STR loci columns recognised.")
    results["loci_used"] = loci_present
    amel_present      = AMELOGENIN in df.columns
    sheet_col_present = "_sheet" in df.columns
    kit_col_present   = "_kit" in df.columns

    id_counts = df[SAMPLE_ID_COL].astype(str).str.strip().value_counts()
    dup_ids   = id_counts[id_counts > 1].index.tolist()
    if dup_ids:
        results["warnings"].append(
            f"<b>Duplicate SampleIDs</b> — only last occurrence used: "
            f"{', '.join(str(x) for x in dup_ids[:10])}"
            f"{'…' if len(dup_ids)>10 else ''}."
        )

    profiles = {}
    for _, row in df.iterrows():
        sid = str(row[SAMPLE_ID_COL]).strip()
        if not sid or sid.lower() in ("nan","none"): continue
        pd_data = {}
        for locus in loci_present:
            pd_data[locus] = parse_alleles(row.get(locus,""))
        if amel_present:
            amel = parse_alleles(row.get(AMELOGENIN,""))
            pd_data[AMELOGENIN] = amel
            g = infer_gender(amel)
            if g == "Uncertain":
                results["warnings"].append(
                    f"<b>{sid}</b>: Amelogenin ambiguous — gender Uncertain.")
        else:
            g = "Unknown"
        pd_data["Gender"] = g
        pd_data["_sheet"] = str(row.get("_sheet","")) if sheet_col_present else ""
        pd_data["_kit"]   = str(row.get("_kit",""))   if kit_col_present   else ""
        profiles[sid] = pd_data
    results["profiles"] = profiles

    if cancel_check(): results["cancelled"]=True; return results

    progress_cb("Mixture screening", 10, time.time()-t0)
    mixture_ids, mixture_flagged = set(), {}
    for sid, data in profiles.items():
        flagged = [l for l in loci_present if len(set(data.get(l,()))) > 2]
        if flagged:
            data["Gender"] = "N/A (Mixture)"
            mixture_ids.add(sid)
            mixture_flagged[sid] = flagged
    results["mixtures"] = {sid: {"profile": profiles[sid],
                                  "flagged_loci": mixture_flagged[sid]}
                           for sid in mixture_ids}
    if mixture_ids:
        results["warnings"].append(
            "⚠ Mixture screening flags profiles with &gt;2 alleles at any locus. "
            "Two-contributor mixtures sharing alleles at every locus will not be "
            "detected automatically — manual electropherogram review required."
        )
    valid_ids = set(profiles.keys()) - mixture_ids
    if cancel_check(): results["cancelled"]=True; return results

    progress_cb("Duplicate profile detection", 20, time.time()-t0)
    dup_loci    = loci_present + ([AMELOGENIN] if amel_present else [])
    hash_groups = defaultdict(list)
    for sid in valid_ids:
        hash_groups[profile_fingerprint(sid, profiles, dup_loci)].append(sid)
    dup_sets = [sorted(g) for g in hash_groups.values() if len(g) > 1]
    results["duplicates"] = dup_sets
    dup_remove = set()
    for ds in dup_sets: dup_remove.update(ds[1:])
    rel_ids = valid_ids - dup_remove
    if cancel_check(): results["cancelled"]=True; return results

    progress_cb("Trio relationship analysis", 35, time.time()-t0)
    profile_list = sorted(rel_ids)
    pot_parents  = defaultdict(list)
    for child_id in profile_list:
        if cancel_check(): results["cancelled"]=True; return results
        for parent_id in profile_list:
            if child_id == parent_id: continue
            ok,_,_ = check_duo(child_id, parent_id, profiles, loci_present, max_mm)
            if ok: pot_parents[child_id].append(parent_id)

    found_trios, children_in_trios = [], set()
    for child_id in [c for c,p in pot_parents.items() if len(p) >= 2]:
        if child_id in children_in_trios: continue
        if cancel_check(): results["cancelled"]=True; return results
        pars     = pot_parents[child_id]
        males    = [p for p in pars if profiles[p]["Gender"]=="Male"]
        females  = [p for p in pars if profiles[p]["Gender"]=="Female"]
        unknowns = [p for p in pars if profiles[p]["Gender"] in ("Unknown","Uncertain")]
        pairs = []
        for fi in females:
            for mi in males: pairs.append({"mother":fi,"father":mi})
        for u1,u2 in itertools.combinations(unknowns,2):
            pairs.append({"mother":u1,"father":u2})
            pairs.append({"mother":u2,"father":u1})
        for u in unknowns:
            for fi in females: pairs.append({"mother":fi,"father":u})
            for mi in males:   pairs.append({"mother":u,"father":mi})
        for pair in pairs:
            m_id, f_id = pair["mother"], pair["father"]
            ok,mm,mm_loci = check_trio(child_id,m_id,f_id,profiles,loci_present,max_mm)
            if ok and child_id not in children_in_trios:
                found_trios.append(dict(
                    child=child_id, mother=m_id, father=f_id,
                    child_gender=profiles[child_id]["Gender"],
                    mother_gender=profiles[m_id]["Gender"],
                    father_gender=profiles[f_id]["Gender"],
                    child_sheet=profiles[child_id].get("_sheet",""),
                    mother_sheet=profiles[m_id].get("_sheet",""),
                    father_sheet=profiles[f_id].get("_sheet",""),
                    child_kit=profiles[child_id].get("_kit",""),
                    mother_kit=profiles[m_id].get("_kit",""),
                    father_kit=profiles[f_id].get("_kit",""),
                    mismatches=mm, mismatch_loci=mm_loci
                ))
                children_in_trios.add(child_id)
                break

    found_trios.sort(key=lambda x: x["child"])
    results["trios"] = found_trios
    ids_in_trios = (
        {t["child"]  for t in found_trios} |
        {t["mother"] for t in found_trios} |
        {t["father"] for t in found_trios}
    )
    if cancel_check(): results["cancelled"]=True; return results

    progress_cb("Duo relationship analysis", 70, time.time()-t0)
    trio_pairs = set()
    for t in found_trios:
        trio_pairs.add(tuple(sorted((t["father"],t["child"]))))
        trio_pairs.add(tuple(sorted((t["mother"],t["child"]))))
    found_duos, assigned_duo = [], set()
    for p1_id in sorted(rel_ids - children_in_trios):
        if p1_id in assigned_duo: continue
        if cancel_check(): results["cancelled"]=True; return results
        best, best_mm, best_loci = None, max_mm+1, []
        for p2_id in sorted(rel_ids - assigned_duo - {p1_id}):
            if tuple(sorted((p1_id,p2_id))) in trio_pairs: continue
            ok,mm,mm_loci = check_duo(p1_id,p2_id,profiles,loci_present,max_mm)
            if ok and mm < best_mm:
                best_mm, best, best_loci = mm, p2_id, mm_loci
        if best:
            found_duos.append(dict(
                profile1=p1_id, profile2=best,
                profile1_gender=profiles[p1_id]["Gender"],
                profile2_gender=profiles[best]["Gender"],
                profile1_sheet=profiles[p1_id].get("_sheet",""),
                profile2_sheet=profiles[best].get("_sheet",""),
                profile1_kit=profiles[p1_id].get("_kit",""),
                profile2_kit=profiles[best].get("_kit",""),
                mismatches=best_mm, mismatch_loci=best_loci
            ))
            assigned_duo.add(p1_id)

    found_duos.sort(key=lambda x: x["profile1"])
    results["duos"] = found_duos
    ids_in_duos = (
        {d["profile1"] for d in found_duos} |
        {d["profile2"] for d in found_duos if d["profile2"] not in ids_in_trios}
    )

    progress_cb("Finalising individual profiles", 90, time.time()-t0)
    remaining = rel_ids - ids_in_trios - ids_in_duos
    results["individuals"] = [
        {"id":sid,
         "gender":profiles[sid]["Gender"],
         "sheet":profiles[sid].get("_sheet",""),
         "kit":profiles[sid].get("_kit","")}
        for sid in sorted(remaining)
    ]
    progress_cb("Generating report", 98, time.time()-t0)
    return results

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — REPORT BUILDER
# ─────────────────────────────────────────────────────────────────────────────

def gender_pill(g):
    cfg = {
        "Male":      (C["male"],    "#dce8ff", "♂ Male"),
        "Female":    (C["female"],  "#ffdce8", "♀ Female"),
        "Uncertain": (C["unknown"], "#e0e0e0", "? Uncertain"),
        "Unknown":   (C["unknown"], "#e0e0e0", "? Unknown"),
    }
    fg, bg, label = cfg.get(g, (C["unknown"], "#e0e0e0", g))
    return (f"<span style='background:{bg};color:{fg};border:1px solid {fg};"
            f"border-radius:3px;padding:1px 7px;font-size:11px;"
            f"font-weight:bold;font-family:monospace'>{label}</span>")

def sheet_kit_tag(sheet, kit):
    parts = []
    if sheet: parts.append(sheet)
    if kit:   parts.append(kit)
    if not parts: return ""
    label = " · ".join(parts)
    return (f"<span style='font-size:10px;color:{C['muted']};"
            f"font-family:monospace;margin-left:4px'>[{label}]</span>")

def cross_sheet_note_trio(t):
    sheets = [t.get("child_sheet",""), t.get("mother_sheet",""),
              t.get("father_sheet","")]
    kits   = [t.get("child_kit",""),  t.get("mother_kit",""),
              t.get("father_kit","")]
    unique_sheets = set(s for s in sheets if s)
    if len(unique_sheets) <= 1: return ""
    detail = (
        f"Father [{t.get('father_sheet','')} · {t.get('father_kit','')}], "
        f"Mother [{t.get('mother_sheet','')} · {t.get('mother_kit','')}], "
        f"Child [{t.get('child_sheet','')} · {t.get('child_kit','')}]"
    )
    return (f"<div style='font-size:11px;color:{C['amber']};margin-top:6px;"
            f"padding:4px 8px;background:#fff8e1;border-radius:3px'>"
            f"⚠ Cross-sheet trio: {detail} — verify loci compatibility "
            f"across kits before reporting.</div>")

def cross_sheet_note_duo(d):
    s1, s2 = d.get("profile1_sheet",""), d.get("profile2_sheet","")
    k1, k2 = d.get("profile1_kit",""),   d.get("profile2_kit","")
    if s1 == s2: return ""
    return (f"<div style='font-size:11px;color:{C['amber']};margin-top:6px;"
            f"padding:4px 8px;background:#fff8e1;border-radius:3px'>"
            f"⚠ Cross-sheet duo: Profile A [{s1} · {k1}], "
            f"Profile B [{s2} · {k2}] — verify loci compatibility "
            f"across kits before reporting.</div>")

def build_report(results, file_name, elapsed, max_mm,
                 is_partial=False, sheet_summary=None):
    profiles  = results["profiles"]
    loci_used = results["loci_used"]
    ts        = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    n_mix  = len(results["mixtures"])
    n_dup  = len(results["duplicates"])
    n_trio = len(results["trios"])
    n_duo  = len(results["duos"])
    n_ind  = len(results["individuals"])
    n_tot  = len(profiles)
    missing_loci = [l for l in STR_LOCI if l not in loci_used]

    # Sheet breakdown for header
    sheet_info = ""
    if sheet_summary and len(sheet_summary) > 0:
        breakdown = ", ".join(
            f"{s['sheet_name']}: {s['n_profiles']}"
            for s in sheet_summary if s["n_profiles"] > 0
        )
        n_sheets = len([s for s in sheet_summary if s["n_profiles"] > 0])
        sheet_info = (f" &nbsp;·&nbsp; <b>Sheets:</b> {n_sheets} "
                      f"({breakdown})")

    def nav_badge(label, count, anchor, hit_color):
        color = hit_color if count > 0 else C["green"]
        return (f"<a href='#{anchor}' style='text-decoration:none'>"
                f"<span style='background:{color};color:#fff;border-radius:4px;"
                f"padding:5px 12px;font-size:12px;font-weight:bold;"
                f"white-space:nowrap;display:inline-block;margin:2px 4px'>"
                f"{label} <span style='background:rgba(255,255,255,0.25);"
                f"border-radius:3px;padding:1px 7px'>{count}</span></span></a>")

    nav_bar = (
        f"<div id='top'></div>"
        f"<div style='position:sticky;top:0;z-index:100;background:{C['navy']};"
        f"padding:8px 16px;display:flex;flex-wrap:wrap;align-items:center;"
        f"gap:4px;border-bottom:2px solid {C['teal']}'>"
        f"<span style='color:#fff;font-weight:bold;font-size:13px;margin-right:8px'>"
        f"🧬 {APP_TITLE}</span>"
        f"{nav_badge('Mixtures',  n_mix,  'sec-mix',  C['amber'])}"
        f"{nav_badge('Duplicates',n_dup,  'sec-dup',  C['slate'])}"
        f"{nav_badge('Trios',     n_trio, 'sec-trio', C['teal'])}"
        f"{nav_badge('Duos',      n_duo,  'sec-duo',  C['green'])}"
        f"{nav_badge('Unrelated', n_ind,  'sec-ind',  C['muted'])}"
        f"<span style='margin-left:auto;display:flex;gap:8px'>"
        f"<button onclick='toggleAll()' id='toggle-btn' style='background:{C['slate']};"
        f"color:#fff;border:none;border-radius:4px;padding:5px 14px;"
        f"font-size:12px;cursor:pointer;font-weight:bold'>⊞ Expand All</button>"
        f"<button onclick='window.print()' style='background:{C['green']};color:#fff;"
        f"border:none;border-radius:4px;padding:5px 14px;font-size:12px;"
        f"cursor:pointer;font-weight:bold'>🖨 Print / Save PDF</button>"
        f"<a href='#top' style='background:{C['navy']};color:#fff;"
        f"border:1px solid #fff;border-radius:4px;padding:5px 14px;"
        f"font-size:12px;text-decoration:none;font-weight:bold'>↑ Top</a>"
        f"</span></div>"
    )

    pdf_bar = (
        f"<div style='background:#fff8e1;border-left:4px solid #f0ad4e;"
        f"padding:10px 16px;font-size:12px;color:{C['text']};margin:10px 0'>"
        f"🖨 <b>To save as PDF:</b> Click <b>Print / Save PDF</b> above "
        f"(or press <b>Ctrl+P</b> / <b>Cmd+Shift+P</b> on Mac) → "
        f"choose <b>Save as PDF</b>. All sections expand automatically when printing."
        f"</div>"
    )

    partial_banner = ""
    if is_partial:
        partial_banner = (
            f"<div style='background:#fff3cd;border-left:5px solid #f0ad4e;"
            f"padding:10px 16px;margin:10px 0;font-size:13px'>"
            f"<b>⚠ Partial Report</b> — Analysis cancelled. "
            f"Only completed phases shown.</div>"
        )

    all_warns = results.get("col_warnings",[]) + results["warnings"]
    warn_html = ""
    if all_warns:
        items = "".join(f"<li style='margin-bottom:4px'>{w}</li>"
                        for w in all_warns)
        warn_html = (
            f"<div style='background:#fff8e1;border-left:4px solid #f0ad4e;"
            f"padding:10px 16px;border-radius:3px;margin:10px 0;font-size:12px'>"
            f"<b>⚠ Analysis Warnings</b>"
            f"<ul style='margin:6px 0 0;padding-left:18px'>{items}</ul></div>"
        )

    # Kit warnings block
    kit_warn_html = ""
    if sheet_summary:
        kit_warns = []
        for s in sheet_summary:
            if s.get("absent_core"):
                kit_warns.append(
                    f"<b>{s['sheet_name']} ({s['kit']}):</b> {s['kit_notes']}"
                )
            elif s.get("kit_notes") and "Penta" in s.get("kit_notes",""):
                kit_warns.append(
                    f"<b>{s['sheet_name']} ({s['kit']}):</b> {s['kit_notes']}"
                )
        if kit_warns:
            items = "".join(f"<li style='margin-bottom:4px'>{w}</li>"
                            for w in kit_warns)
            kit_warn_html = (
                f"<div style='background:#fff8e1;border-left:4px solid #f0ad4e;"
                f"padding:10px 16px;border-radius:3px;margin:10px 0;font-size:12px'>"
                f"<b>🧪 Kit Compatibility Notes</b>"
                f"<ul style='margin:6px 0 0;padding-left:18px'>{items}</ul></div>"
            )

    # Sheet summary table in report
    sheet_table_html = ""
    if sheet_summary and len(sheet_summary) > 1:
        rows = "".join(
            f"<tr style='background:{'#fff' if i%2==0 else C['pale']}'>"
            f"<td style='padding:5px 10px;border:1px solid {C['border']};"
            f"font-family:monospace'>{s['sheet_name']}</td>"
            f"<td style='padding:5px 10px;border:1px solid {C['border']};text-align:right'>"
            f"{s['n_profiles']}</td>"
            f"<td style='padding:5px 10px;border:1px solid {C['border']}'>"
            f"<b>{s['kit']}</b><br>"
            f"<span style='font-size:11px;color:{C['muted']}'>"
            f"{s['kit_manufacturer']}</span></td>"
            f"<td style='padding:5px 10px;border:1px solid {C['border']};text-align:right'>"
            f"{s['loci_count']}/{len(STR_LOCI)}</td>"
            f"<td style='padding:5px 10px;border:1px solid {C['border']};font-size:11px'>"
            f"{s['status']}</td></tr>"
            for i,s in enumerate(sheet_summary)
        )
        sheet_table_html = (
            f"<div class='seg'><details>"
            f"<summary style='color:{C['navy']}'>Input Sheets Summary"
            f"<span style='background:{C['slate']};color:#fff;border-radius:3px;"
            f"padding:1px 8px;font-size:12px;margin-left:8px;font-family:monospace'>"
            f"{len(sheet_summary)} sheets</span></summary>"
            f"<div class='seg-body'>"
            f"<table style='width:100%;border-collapse:collapse;font-size:13px'>"
            f"<tr style='background:{C['navy']};color:#fff'>"
            f"<th style='padding:5px 10px;text-align:left'>Sheet</th>"
            f"<th style='padding:5px 10px;text-align:right'>Profiles</th>"
            f"<th style='padding:5px 10px;text-align:left'>Kit Detected</th>"
            f"<th style='padding:5px 10px;text-align:right'>Loci</th>"
            f"<th style='padding:5px 10px;text-align:left'>Status</th></tr>"
            f"{rows}</table></div></details></div>"
        )

    def mrow(label, val, shade):
        bg = f"background:{C['pale']};" if shade else ""
        return (f"<tr><td style='{bg}padding:5px 10px;border:1px solid {C['border']}'>"
                f"{label}</td><td style='{bg}padding:5px 10px;"
                f"border:1px solid {C['border']};text-align:right;"
                f"font-weight:bold'>{val}</td></tr>")

    n_sheets_used = len([s for s in (sheet_summary or []) if s["n_profiles"] > 0])
    perf_html = (
        f"<table style='width:100%;border-collapse:collapse;font-size:13px;margin-top:8px'>"
        f"<tr style='background:{C['navy']};color:#fff'>"
        f"<th style='padding:6px 10px;text-align:left'>Metric</th>"
        f"<th style='padding:6px 10px;text-align:right'>Value</th></tr>"
        f"{mrow('Total profiles processed', n_tot, False)}"
        f"{mrow('Sheets processed', n_sheets_used if n_sheets_used else '1', True)}"
        f"{mrow(f'Loci analysed', f'{len(loci_used)} / {len(STR_LOCI)}', False)}"
        f"{mrow('Missing loci', ', '.join(missing_loci) if missing_loci else 'None', True)}"
        f"{mrow('Mismatch threshold (Trio &amp; Duo)', max_mm, False)}"
        f"{mrow('Mixtures detected', n_mix, True)}"
        f"{mrow('Duplicate sets', n_dup, False)}"
        f"{mrow('Trios identified', n_trio, True)}"
        f"{mrow('Duos identified', n_duo, False)}"
        f"{mrow('Unrelated individuals', n_ind, True)}"
        f"{mrow('Analysis time', f'{elapsed:.2f} seconds', False)}"
        f"</table>"
    )

    def seg(anchor, title, count, body, hit_color, note=""):
        color  = hit_color if count > 0 else C["green"]
        badge  = (f"<span style='background:{color};color:#fff;border-radius:3px;"
                  f"padding:1px 8px;font-size:12px;margin-left:8px;"
                  f"font-family:monospace'>{count} found</span>")
        note_h = (f"<div style='font-size:12px;color:{C['muted']};font-style:italic;"
                  f"margin-bottom:8px;padding:6px 10px;background:{C['light']};"
                  f"border-radius:3px'>{note}</div>") if note else ""
        return (f"<div id='{anchor}' class='seg'><details>"
                f"<summary style='color:{C['navy']}'>{title}{badge}</summary>"
                f"<div class='seg-body'>{note_h}{body}</div>"
                f"</details></div>")

    def row_wrap(cells, cols=3):
        html = ""
        for i,c in enumerate(cells):
            if i % cols == 0: html += "<tr>"
            html += c
            if (i+1) % cols == 0 or i == len(cells)-1: html += "</tr>"
        return html

    def grid(cells):
        return f"<table class='rtable'>{row_wrap(cells)}</table>"

    # mixtures
    if results["mixtures"]:
        cells = []
        for mid, mdata in sorted(results["mixtures"].items()):
            fl = ", ".join(mdata["flagged_loci"])
            sh = mdata["profile"].get("_sheet","")
            kt = mdata["profile"].get("_kit","")
            cells.append(
                f"<td><b style='font-family:monospace'>{mid}</b>"
                f"{sheet_kit_tag(sh,kt)}<br>"
                f"<span style='font-size:11px;color:{C['amber']}'>"
                f"Extra alleles at: {fl}</span></td>"
            )
        mix_body = grid(cells)
    else:
        mix_body = f"<p style='color:{C['green']}'>✅ No mixture profiles detected.</p>"

    # duplicates
    if results["duplicates"]:
        items = ""
        for i,ds in enumerate(results["duplicates"]):
            g = profiles[ds[0]]["Gender"]
            ids_str = " &nbsp;·&nbsp; ".join(
                f"<code>{s}</code>"
                f"{sheet_kit_tag(profiles[s].get('_sheet',''),profiles[s].get('_kit',''))}"
                for s in ds
            )
            items += (f"<li style='margin-bottom:8px'>"
                      f"<b>Set {i+1}:</b> {ids_str} &nbsp;{gender_pill(g)}</li>")
        dup_body = f"<ul style='line-height:2;padding-left:18px'>{items}</ul>"
    else:
        dup_body = f"<p style='color:{C['green']}'>✅ No duplicate profiles detected.</p>"

    # trios
    trio_note = (
        "Father and Mother labels are assigned based on Amelogenin gender inference. "
        "Where gender is Uncertain, assignments are tentative and should be verified."
    )
    if results["trios"]:
        cells = []
        for t in results["trios"]:
            mm_d = ""
            if t["mismatches"] > 0:
                mm_d = (f"<div style='font-size:11px;color:#1a2a35;margin-top:4px;"
                        f"border-top:1px dashed #ccc;padding-top:4px'>"
                        f"<b style='color:{C['red']}'>Mismatch loci:</b> "
                        f"{', '.join(t['mismatch_loci'])}</div>")
            cs_note = cross_sheet_note_trio(t)
            ped = (
                f"<div style='font-size:12px;line-height:1.9;margin-top:8px;"
                f"border-top:1px solid {C['border']};padding-top:8px'>"
                f"<b>Father:</b> <code>{t['father']}</code>"
                f"{sheet_kit_tag(t.get('father_sheet',''),t.get('father_kit',''))} "
                f"{gender_pill(t['father_gender'])}<br>"
                f"<b>Mother:</b> <code>{t['mother']}</code>"
                f"{sheet_kit_tag(t.get('mother_sheet',''),t.get('mother_kit',''))} "
                f"{gender_pill(t['mother_gender'])}<br>"
                f"<span style='color:{C['border']}'>──────────────────</span><br>"
                f"<b>Child &nbsp;:</b> <code>{t['child']}</code>"
                f"{sheet_kit_tag(t.get('child_sheet',''),t.get('child_kit',''))} "
                f"{gender_pill(t['child_gender'])}<br>"
                f"<b>Mismatches:</b> {t['mismatches']}"
                f"{mm_d}{cs_note}</div>"
            )
            cells.append(f"<td>{ped}</td>")
        trio_body = grid(cells)
    else:
        trio_body = f"<p style='color:{C['green']}'>✅ No trio relationships identified.</p>"

    # duos
    duo_note = (
        "Kinship relationship detected. Directionality (parent/child) cannot be "
        "determined from a duo alone — further investigation required."
    )
    if results["duos"]:
        cells = []
        for d in results["duos"]:
            mm_d = ""
            if d["mismatches"] > 0:
                mm_d = (f"<div style='font-size:11px;color:#1a2a35;margin-top:4px;"
                        f"border-top:1px dashed #ccc;padding-top:4px'>"
                        f"<b style='color:{C['red']}'>Mismatch loci:</b> "
                        f"{', '.join(d['mismatch_loci'])}</div>")
            cs_note = cross_sheet_note_duo(d)
            ped = (
                f"<div style='font-size:12px;line-height:1.9;margin-top:8px;"
                f"border-top:1px solid {C['border']};padding-top:8px'>"
                f"<b>Profile A:</b> <code>{d['profile1']}</code>"
                f"{sheet_kit_tag(d.get('profile1_sheet',''),d.get('profile1_kit',''))} "
                f"{gender_pill(d['profile1_gender'])}<br>"
                f"<span style='color:{C['teal']};font-size:16px;margin-left:8px'>"
                f"⟷</span><br>"
                f"<b>Profile B:</b> <code>{d['profile2']}</code>"
                f"{sheet_kit_tag(d.get('profile2_sheet',''),d.get('profile2_kit',''))} "
                f"{gender_pill(d['profile2_gender'])}<br>"
                f"<b>Mismatches:</b> {d['mismatches']}"
                f"{mm_d}{cs_note}</div>"
            )
            cells.append(f"<td>{ped}</td>")
        duo_body = grid(cells)
    else:
        duo_body = f"<p style='color:{C['green']}'>✅ No duo relationships identified.</p>"

    # individuals
    if results["individuals"]:
        cells = [
            f"<td style='text-align:center'><code>{ind['id']}</code>"
            f"{sheet_kit_tag(ind.get('sheet',''),ind.get('kit',''))}<br>"
            f"{gender_pill(ind['gender'])}</td>"
            for ind in results["individuals"]
        ]
        ind_body = grid(cells)
    else:
        ind_body = (f"<p style='color:{C['green']}'>"
                    f"✅ All profiles assigned to a relationship category.</p>")

    css = (
        f"<style>"
        f"*{{box-sizing:border-box}}"
        f"body{{font-family:Arial,sans-serif;color:{C['text']};margin:0;"
        f"padding:0 0 40px 0;line-height:1.5;background:#fff}}"
        f".report-body{{max-width:1100px;margin:0 auto;padding:20px 24px}}"
        f"h1{{color:{C['navy']};border-bottom:3px solid {C['teal']};"
        f"padding-bottom:6px;font-size:20px;margin-bottom:4px}}"
        f".seg{{border:1px solid {C['border']};border-radius:4px;"
        f"margin-bottom:10px;padding:10px 14px;background:{C['pale']}}}"
        f"summary{{cursor:pointer;font-weight:bold;font-size:14px;"
        f"list-style:none;padding:4px 0;user-select:none}}"
        f"summary::-webkit-details-marker{{display:none}}"
        f"summary::before{{content:'▶ ';font-size:10px;color:{C['muted']}}}"
        f"details[open] summary::before{{content:'▼ '}}"
        f".seg-body{{padding-top:10px}}"
        f".rtable{{width:100%;border-collapse:collapse;margin-top:8px}}"
        f".rtable td{{border:1px solid {C['border']};padding:10px;"
        f"background:#fff;width:32%;vertical-align:top}}"
        f".footer-bar{{display:flex;gap:12px;align-items:center;"
        f"justify-content:flex-end;margin-top:30px;"
        f"padding-top:14px;border-top:1px solid {C['border']}}}"
        f".footer-bar button,.footer-bar a{{background:{C['navy']};color:#fff;"
        f"border:none;border-radius:4px;padding:8px 18px;font-size:13px;"
        f"cursor:pointer;font-weight:bold;text-decoration:none;display:inline-block}}"
        f".footer-bar .print-btn{{background:{C['green']}}}"
        f".disclaimer{{margin-top:20px;padding:12px 16px;"
        f"background:{C['light']};border-left:4px solid {C['navy']};"
        f"border-radius:3px;font-size:11px;color:{C['muted']}}}"
        f".report-footer{{font-size:10px;color:{C['muted']};margin-top:12px;"
        f"border-top:1px solid {C['border']};padding-top:8px}}"
        f"@media print{{"
        f".no-print{{display:none!important}}"
        f"body{{margin:0;padding:0}}"
        f".report-body{{padding:10mm}}"
        f".seg{{page-break-inside:avoid;border:1px solid #999}}"
        f"details{{display:block!important}}"
        f"summary::before{{content:''!important}}"
        f".rtable td{{page-break-inside:avoid}}"
        f"@page{{margin:15mm;"
        f"@bottom-right{{content:'Page ' counter(page) ' of ' counter(pages);"
        f"font-size:9pt;color:#666}}}}}}"
        f"</style>"
        f"<script>"
        f"var allOpen=false;"
        f"function toggleAll(){{"
        f"var ds=document.querySelectorAll('.seg details');"
        f"allOpen=!allOpen;"
        f"ds.forEach(function(d){{d.open=allOpen;}});"
        f"document.getElementById('toggle-btn').textContent="
        f"allOpen?'⊟ Collapse All':'⊞ Expand All';}}"
        f"</script>"
    )

    loci_str = " · ".join(loci_used)
    miss_str = (f"<br><span style='color:{C['red']}'>"
                f"⚠ Missing loci: {', '.join(missing_loci)}</span>"
                if missing_loci else "")

    html = (
        f"<!DOCTYPE html><html><head><meta charset='utf-8'>"
        f"<title>PyKinshipID Report — {ts}</title>{css}</head><body>"
        f"{nav_bar}<div class='report-body'>"
        f"{pdf_bar}{partial_banner}"
        f"<h1>🧬 {APP_TITLE} — Analysis Report</h1>"
                f"<p style='font-size:13px'><b>File:</b> {file_name}{sheet_info} &nbsp;·&nbsp; "
        f"<b>Profiles:</b> {n_tot} &nbsp;·&nbsp; "
        f"<b>Generated:</b> {ts} &nbsp;·&nbsp; "
        f"<b>Mismatch threshold:</b> {max_mm}</p>"
        f"<p style='font-size:12px;color:{C['muted']}'>"
        f"<b>Loci ({len(loci_used)}/{len(STR_LOCI)}):</b> {loci_str}{miss_str}</p>"
        f"{warn_html}{kit_warn_html}{sheet_table_html}"
        f"<div class='seg'><details>"
        f"<summary style='color:{C['navy']}'>Performance Metrics</summary>"
        f"<div class='seg-body'>{perf_html}</div></details></div>"
        f"<div style='text-align:right;margin:2px 0 8px'><a href='#top' style='font-size:11px;color:#546e7a;text-decoration:none'>&#8679; Top</a></div>"
        f"{seg('sec-mix',  '1. Mixture Profiles',              n_mix,  mix_body,  C['amber'])}"
        f"<div style='text-align:right;margin:2px 0 8px'><a href='#top' style='font-size:11px;color:#546e7a;text-decoration:none'>&#8679; Top</a></div>"
        f"{seg('sec-dup',  '2. Duplicate Profiles',             n_dup,  dup_body,  C['slate'])}"
        f"<div style='text-align:right;margin:2px 0 8px'><a href='#top' style='font-size:11px;color:#546e7a;text-decoration:none'>&#8679; Top</a></div>"
        f"{seg('sec-trio', '3. Trio Relationships',             n_trio, trio_body, C['teal'],  trio_note)}"
        f"<div style='text-align:right;margin:2px 0 8px'><a href='#top' style='font-size:11px;color:#546e7a;text-decoration:none'>&#8679; Top</a></div>"
        f"{seg('sec-duo',  '4. Duo Relationships',              n_duo,  duo_body,  C['green'], duo_note)}"
        f"<div style='text-align:right;margin:2px 0 8px'><a href='#top' style='font-size:11px;color:#546e7a;text-decoration:none'>&#8679; Top</a></div>"
        f"{seg('sec-ind',  '5. Unrelated / Individual Profiles',n_ind,  ind_body,  C['muted'])}"
        f"<div class='disclaimer'>{DISCLAIMER}</div>"
        f"<div class='footer-bar no-print'>"
        f"<button class='print-btn' onclick='window.print()'>🖨 Print / Save PDF</button>"
        f"<a href='#top'>↑ Top</a></div>"
        f"<div class='report-footer'>{VERSION} &nbsp;·&nbsp; "
        f"Report generated: {ts} &nbsp;·&nbsp; For screening use only.</div>"
        f"</div></body></html>"
    )
    return html

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — SAMPLE DATA
# ─────────────────────────────────────────────────────────────────────────────

def make_sample_df():
    import pandas as pd
    rows = [
        ["TRIO_1_Father","X,Y","10,14","26,26","10,11","11,13","19,21","8,8","12,12","7,13","15,22","11,18","21,22","8,9","23,23","15,15","19,23"],
        ["TRIO_1_Mother","X,X","15,20","25,38","11,12","7,7","12,12","6,11","8,9","10,11","23,24","9,16","11,21","7,9","22,26","12,17","34,37"],
        ["TRIO_1_Child","X,Y","10,15","26,38","10,12","7,13","12,21","6,8","8,12","10,13","15,24","16,18","21,21","7,8","23,26","15,17","23,37"],
        ["TRIO_2_Father","X,Y","11,11","26,30","10,10","7,13","12,22","9,9","16,16","10,10","18,27","11,12","11,18","7,7","22,26","14,14","29,33"],
        ["TRIO_2_Mother","X,X","13,16","38,38","11,12","6,10","14,21","12,12","13,16","14,14","21,21","12,17","12,15","6,6","11,15","12,12","33,39"],
        ["TRIO_2_Child","X,X","11,13","30,38","10,11","6,7","14,22","9,12","13,16","10,14","21,27","12,12","12,18","6,7","15,26","12,14","29,39"],
        ["TRIO_3_Father","X,Y","12,14","28,31","9,12","8,12","15,18","7,9.3","11,14","9,12","17,23","13,14","14,19","8,11","15,18","11,14","21,25"],
        ["TRIO_3_Mother","X,X","10,13","29,33","10,11","7,9","16,17","6,8","12,15","11,13","20,22","12,15","16,18","7,10","16,20","12,13","22,24"],
        ["TRIO_3_Child","X,X","12,10","28,29","9,10","8,7","15,16","9.3,6","11,12","9,11","17,20","14,12","14,16","8,7","15,16","11,12","21,22"],
        ["TRIO_4_Father","X,Y","13,15","27,32","8,11","9,11","16,20","8,9","10,13","8,11","16,24","14,16","15,17","9,12","14,17","10,13","20,26"],
        ["TRIO_4_Mother","X,X","11,14","30,34","9,10","8,10","17,19","7,8","11,14","10,12","19,21","13,15","17,20","8,11","15,18","11,14","23,25"],
        ["TRIO_4_Child","X,Y","13,11","27,30","8,9","9,8","16,17","8,7","10,11","8,10","16,19","14,13","15,17","9,8","14,15","10,11","20,23"],
        ["DUO_1_ProfileA","X,X","12,15","29,35","9,11","8,11","16,19","7,9","11,14","9,12","18,22","12,15","16,19","8,10","15,19","11,14","22,26"],
        ["DUO_1_ProfileB","X,Y","12,13","29,36","9,10","8,9","16,18","7,8","11,12","9,10","18,20","12,13","16,17","8,9","15,16","11,12","22,23"],
        ["DUO_2_ProfileA","X,Y","10,14","27,31","8,10","9,12","15,18","8,9","12,15","10,13","17,21","13,16","14,17","9,11","16,20","12,15","21,24"],
        ["DUO_2_ProfileB","X,X","10,11","27,29","8,9","9,10","15,17","8,7","12,13","10,11","17,18","13,14","14,15","9,8","16,17","12,13","21,22"],
        ["DUP_Sample_A","X,Y","14,17","30,33","11,12","10,12","17,20","9,10","13,15","11,13","19,23","14,16","18,21","9,11","17,20","13,15","24,28"],
        ["DUP_Sample_B","X,Y","14,17","30,33","11,12","10,12","17,20","9,10","13,15","11,13","19,23","14,16","18,21","9,11","17,20","13,15","24,28"],
        ["MIX_001","X,X","13,14,15","28,31","10,11","9,11","16,18","7,9","12,14","10,12","18,21","13,15","16,18","8,10","15,18","12,14","22,25"],
    ]
    cols = [SAMPLE_ID_COL, AMELOGENIN] + STR_LOCI
    return pd.DataFrame(rows, columns=cols)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — UI HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _html(v): return widgets.HTML(value=v)
def _btn(desc, color=None, width="auto"):
    return widgets.Button(
        description=desc,
        style={"button_color": color or C["navy"],
               "font_weight":"bold","font_size":"13px"},
        layout=widgets.Layout(width=width, margin="4px 6px 4px 0",
                              min_width="120px")
    )

def card(content, bc=None, bg=None):
    return (f"<div style='border:1px solid {bc or C['border']};border-radius:4px;"
            f"padding:14px 18px;background:{bg or C['pale']};margin-bottom:10px'>"
            f"{content}</div>")

def sec_label(text, note=""):
    n = (f"<span style='font-size:11px;color:{C['muted']};margin-left:8px'>"
         f"{note}</span>") if note else ""
    return (f"<div style='font-size:12px;font-weight:bold;color:{C['navy']};"
            f"border-left:3px solid {C['teal']};padding-left:8px;"
            f"margin:10px 0 4px'>{text}{n}</div>")

def file_details_card(df, source, fname, fsize_kb=None, sheet_summary=None):
    import pandas as pd
    n    = len(df)
    cols = df.columns.tolist()
    loci_found = [l for l in STR_LOCI if l in cols]
    loci_miss  = [l for l in STR_LOCI if l not in cols]
    amel_found = AMELOGENIN in cols
    size_str   = f"{fsize_kb:.1f} KB" if fsize_kb else "—"

    pills = ""
    if amel_found:
        genders = df[AMELOGENIN].apply(
            lambda x: infer_gender(parse_alleles(str(x))))
        gc = genders.value_counts().to_dict()
        pills = " &nbsp;".join(
            f"{gender_pill(g)} {gc.get(g,0)}"
            for g in ["Male","Female","Uncertain"]
        )

    idle_warn = (
        f"<br><span style='color:{C['red']}'>"
        f"⚠ Keep this tab active — Colab disconnects after 30 minutes idle.</span>"
    ) if n > 9000 else ""

    est = estimate_time(n)
    est_html = (
        f"<div style='margin-top:12px;padding:10px 14px;background:#fff8e1;"
        f"border-left:4px solid #f0ad4e;font-size:12px;color:{C['text']};"
        f"border-radius:0 4px 4px 0'>"
        f"⏱ <b>Estimated analysis time: {est}</b>{idle_warn}</div>"
    )

    # Sheet summary table — always shown, even for single sheet
    sheet_html = ""
    if sheet_summary:
        rows = "".join(
            f"<tr style='background:{'#ffffff' if i%2==0 else C['pale']}'>"
            f"<td style='padding:6px 10px;border:1px solid {C['border']};"
            f"font-family:monospace;color:{C['navy']};font-weight:bold'>{s['sheet_name']}</td>"
            f"<td style='padding:6px 10px;border:1px solid {C['border']};"
            f"text-align:right;font-weight:bold;color:{C['text']}'>{s['n_profiles']}</td>"
            f"<td style='padding:6px 10px;border:1px solid {C['border']};color:{C['text']}'>"
            f"<b>{s['kit']}</b>"
            f"<span style='font-size:10px;color:{C['muted']};margin-left:4px'>"
            f"{s['kit_manufacturer']}</span></td>"
            f"<td style='padding:6px 10px;border:1px solid {C['border']};"
            f"text-align:right;color:{C['text']}'>{s['loci_count']}/{len(STR_LOCI)}</td>"
            f"<td style='padding:6px 10px;border:1px solid {C['border']};"
            f"font-size:11px;color:{C['text']}'>{s['status']}</td></tr>"
            for i,s in enumerate(sheet_summary)
        )
        sheet_html = (
            f"<div style='margin-top:12px'>"
            f"<b style='font-size:12px;color:{C['navy']}'>"
            f"📊 Sheet Summary ({len(sheet_summary)} sheet"
            f"{'s' if len(sheet_summary)>1 else ''})</b>"
            f"<table style='width:100%;border-collapse:collapse;"
            f"font-size:12px;margin-top:6px;border:1px solid {C['border']}'>"
            f"<tr style='background:{C['navy']};color:#fff'>"
            f"<th style='padding:6px 10px;text-align:left'>Sheet</th>"
            f"<th style='padding:6px 10px;text-align:right'>Profiles</th>"
            f"<th style='padding:6px 10px;text-align:left'>Kit Detected</th>"
            f"<th style='padding:6px 10px;text-align:right'>Loci</th>"
            f"<th style='padding:6px 10px;text-align:left'>Status</th></tr>"
            f"{rows}</table></div>"
        )

    def drow(label, value, shade=False):
        bg = C['light'] if shade else "#ffffff"
        return (
            f"<tr>"
            f"<td style='background:{C['navy']};color:#d0e8f0;padding:6px 12px;"
            f"font-size:12px;font-weight:bold;white-space:nowrap;width:160px'>"
            f"{label}</td>"
            f"<td style='background:{bg};color:{C['text']};padding:6px 12px;"
            f"font-size:13px;border-bottom:1px solid {C['border']}'>"
            f"{value}</td>"
            f"</tr>"
        )

    main_rows = (
        drow("Source",         f"<b>{source}</b>",                          False) +
        drow("File",           f"<code style='font-size:12px;color:{C['navy']}'>{fname}</code>", True) +
        drow("File size",      size_str,                                    False) +
        drow("Total profiles", f"<b style='font-size:18px;color:{C['teal']}'>{n}</b>", True) +
        drow("Loci detected",
             f"<b>{len(loci_found)}/{len(STR_LOCI)}</b>"
             f"<span style='font-size:11px;color:{C['muted']};margin-left:8px'>"
             f"{', '.join(loci_found)}</span>",                             False) +
        (drow("Missing loci",
              f"<span style='color:{C['red']};font-weight:bold'>"
              f"{', '.join(loci_miss)}</span>", True) if loci_miss else "") +
        drow("Amelogenin",
             "✅ Present" if amel_found else
             f"<span style='color:{C['red']}'>❌ Not found</span>",         False if loci_miss else True) +
        (f"<tr>"
         f"<td style='background:{C['navy']};color:#d0e8f0;padding:6px 12px;"
         f"font-size:12px;font-weight:bold;white-space:nowrap'>Gender</td>"
         f"<td style='background:{C['light']};color:{C['text']};padding:6px 12px;"
         f"border-bottom:1px solid {C['border']}'>{pills}</td></tr>"
         if amel_found else "")
    )

    return card(
        f"<b style='color:{C['navy']};font-size:14px'>📂 Loaded Data Summary</b>"
        f"<table style='width:100%;border-collapse:collapse;margin-top:10px;"
        f"border-radius:4px;overflow:hidden;border:1px solid {C['border']}'>"
        f"{main_rows}</table>"
        f"{sheet_html}{est_html}",
        bc=C["teal"], bg="#f0f7f5"
    )

def progress_bar_html(phase, pct, elapsed):
    return (
        f"<div style='border:1px solid {C['border']};border-radius:4px;"
        f"padding:12px 16px;background:{C['pale']}'>"
        f"<div style='font-size:13px;font-weight:bold;color:{C['navy']}'>"
        f"⏳ {phase}</div>"
        f"<div style='background:{C['border']};border-radius:3px;"
        f"height:14px;margin:6px 0'>"
        f"<div style='background:{C['teal']};width:{pct}%;height:14px;"
        f"border-radius:3px'></div></div>"
        f"<div style='font-size:11px;color:{C['muted']};margin-top:2px'>"
        f"{pct}% complete &nbsp;·&nbsp; Elapsed: {elapsed:.1f}s</div></div>"
    )

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — LAYOUT
# ─────────────────────────────────────────────────────────────────────────────

out_file      = widgets.Output()
out_preview   = widgets.Output()
out_progress  = widgets.Output()
out_results   = widgets.Output()
out_download  = widgets.Output()
out_status    = widgets.Output()
out_wipe_conf = widgets.Output()

def make_header(summary_line=""):
    sum_html = (
        f"<div style='font-size:11px;color:#a0c4cc;margin-top:3px'>"
        f"Last run: {summary_line}</div>"
    ) if summary_line else ""
    return (
        f"<div id='pykinshipid-top' style='background:{C['navy']};color:#fff;"
        f"padding:14px 20px;border-radius:4px 4px 0 0'>"
        f"<div style='display:flex;justify-content:space-between;"
        f"align-items:flex-start'>"
        f"<div><span style='font-size:20px;font-weight:bold'>"
        f"🧬 {APP_TITLE}</span>"
        f"</div>"
        f"<span style='font-size:11px;color:#c8dce6;margin-top:4px'>"
        f"Use Wipe Session to clear all data</span></div>"
        f"<div style='font-size:12px;color:#d0e8f0;margin-top:5px'>"
        f"Click Play to identify: "
        f"1. Duplicate DNA Profiles &nbsp;·&nbsp; "
        f"2. Mixture DNA Profiles &nbsp;·&nbsp; "
        f"3. Trios &nbsp;·&nbsp; 4. Duos &nbsp;·&nbsp; "
        f"5. Unrelated DNA Profiles</div>{sum_html}</div>"
        f"<div style='background:#fff8e1;border-left:4px solid #f0ad4e;"
        f"padding:8px 14px;font-size:11px;color:{C['text']};margin-bottom:10px'>"
        f"{DISCLAIMER}</div>"
    )

w_header = _html(make_header())

slider_mm = widgets.IntSlider(
    value=1, min=0, max=2, step=1,
    description="Max mismatches (Trio & Duo):",
    style={"description_width":"210px"},
    layout=widgets.Layout(width="420px")
)

uploader = widgets.FileUpload(
    accept=".xlsx,.csv", multiple=False,
    layout=widgets.Layout(margin="4px 8px 4px 0")
)

def _reset_uploader():
    """
    Replace the FileUpload widget with a fresh instance so the
    displayed filename is cleared. Then swap it into the HBox.
    """
    global uploader
    uploader = widgets.FileUpload(
        accept=".xlsx,.csv", multiple=False,
        layout=widgets.Layout(margin="4px 8px 4px 0")
    )
    uploader.observe(on_upload, names="value")
    # Swap into the HBox that holds [uploader, btn_sample]
    upload_row.children = (uploader, btn_sample)


btn_sample       = _btn("📦 Load Sample Data",  color="#2d5a3d", width="180px")
btn_analyse      = _btn("🔬 Run Analysis",       color=C["navy"], width="160px")
btn_cancel       = _btn("⛔ Cancel Analysis",    color=C["red"],  width="160px")
btn_wipe1        = _btn("🗑 Wipe Session",       color=C["red"],  width="150px")
btn_wipe2        = _btn("🗑 Wipe Session",       color=C["red"],  width="150px")
btn_wipe3        = _btn("🗑 Wipe Session",       color=C["red"],  width="150px")
btn_confirm_wipe = _btn("⚠ Confirm Wipe — this cannot be undone",
                         color="#5a0000", width="320px")
btn_cancel.layout.display = "none"

w_steps = _html(
    f"<div style='border:1px solid {C['teal']};border-radius:4px;"
    f"padding:12px 18px;background:#f0f7f5;margin-bottom:10px'>"
    f"<b style='color:{C['navy']};font-size:11px'>How to use PyKinshipID</b>"
    f"<div style='display:flex;gap:0;margin-top:8px;font-size:10px'>"
    f"<div style='flex:1;text-align:center;padding:6px 4px;"
    f"background:{C['navy']};color:#fff;border-radius:3px 0 0 3px'>"
    f"<div style='font-size:14px;font-weight:bold'>1</div>"
    f"<div>Upload data</div><div style='color:rgba(255,255,255,0.92)'>or load sample</div></div>"
    f"<div style='width:2px;background:{C['teal']}'></div>"
    f"<div style='flex:1;text-align:center;padding:6px 4px;"
    f"background:{C['slate']};color:#fff'>"
    f"<div style='font-size:14px;font-weight:bold'>2</div>"
    f"<div>Set mismatch</div><div style='color:rgba(255,255,255,0.92)'>threshold (0-2)</div></div>"
    f"<div style='width:2px;background:{C['teal']}'></div>"
    f"<div style='flex:1;text-align:center;padding:6px 4px;"
    f"background:{C['teal']};color:#fff'>"
    f"<div style='font-size:14px;font-weight:bold'>3</div>"
    f"<div>Run Analysis</div><div style='color:rgba(255,255,255,0.92)'>click the button</div></div>"
    f"<div style='width:2px;background:{C['teal']}'></div>"
    f"<div style='flex:1;text-align:center;padding:6px 4px;"
    f"background:{C['green']};color:#fff'>"
    f"<div style='font-size:14px;font-weight:bold'>4</div>"
    f"<div>Download report</div><div style='color:rgba(255,255,255,0.92)'>HTML then PDF</div></div>"
    f"<div style='width:2px;background:{C['teal']}'></div>"
    f"<div style='flex:1;text-align:center;padding:6px 4px;"
    f"background:{C['red']};color:#fff;border-radius:0 3px 3px 0'>"
    f"<div style='font-size:14px;font-weight:bold'>5</div>"
    f"<div>Wipe session</div><div style='color:rgba(255,255,255,0.92)'>clear all data</div></div>"
    f"</div></div>"
)

divider = _html(
    f"<hr style='border:none;border-top:1px solid {C['border']};margin:10px 0'>"
)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — WIPE
# ─────────────────────────────────────────────────────────────────────────────

def do_wipe():
    fname = _state.get("file_name","")
    fpath = f"/content/{fname}"
    if fname and os.path.exists(fpath):
        try: os.remove(fpath)
        except: pass
    _state.clear()
    _state["cancel"] = False
    _state["wiped"]  = True
    for o in [out_file,out_preview,out_progress,out_results,
              out_download,out_status,out_wipe_conf]:
        o.clear_output()
    _reset_uploader()
    w_header.value = make_header()
    with out_status:
        display(_html(card(
            f"<b style='color:{C['red']}'>🗑 Session Wiped</b><br>"
            f"<span style='font-size:13px'>"
            f"All loaded data, variables, and results cleared.<br><br>"
            f"<b>For complete assurance:</b> click "
            f"<b>Runtime → Restart Runtime</b> in the Colab menu above.</span>",
            bc=C["red"], bg="#fff5f5"
        )))

def on_wipe_step1(_):
    out_wipe_conf.clear_output()
    with out_wipe_conf:
        display(widgets.HBox([
            _html(f"<span style='color:{C['red']};font-size:13px;"
                  f"font-weight:bold;line-height:36px;margin-right:10px'>"
                  f"Are you sure? All data will be cleared.</span>"),
            btn_confirm_wipe
        ]))

def on_confirm_wipe(_):
    out_wipe_conf.clear_output(); do_wipe()

btn_wipe1.on_click(on_wipe_step1)
btn_wipe2.on_click(on_wipe_step1)
btn_wipe3.on_click(on_wipe_step1)
btn_confirm_wipe.on_click(on_confirm_wipe)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — DATA LOAD HANDLERS
# ─────────────────────────────────────────────────────────────────────────────

def show_preview(df, source, fname, fsize_kb=None, sheet_summary=None):
    import pandas as pd
    out_file.clear_output(); out_preview.clear_output()
    with out_file:
        display(_html(file_details_card(
            df, source, fname, fsize_kb, sheet_summary)))
    with out_preview:
        display(_html(sec_label("First 5 Profiles",
                      "Verify column mapping before running analysis")))
        display(df.head())

def on_load_sample(_):
    for o in [out_file,out_preview,out_progress,out_results,
              out_download,out_status,out_wipe_conf]:
        o.clear_output()
    df = make_sample_df()
    # Build single-sheet summary for sample data
    kit_info = detect_kit(df.columns.tolist())
    sample_summary = [dict(
        sheet_name="Sample Dataset",
        n_profiles=len(df),
        loci_count=len([l for l in STR_LOCI if l in df.columns]),
        loci_found=[l for l in STR_LOCI if l in df.columns],
        kit=kit_info["name"],
        kit_manufacturer=kit_info["manufacturer"],
        kit_notes=kit_info["notes"],
        absent_core=kit_info["absent_core"],
        status=f"✅ {len(df)} profiles · {len([l for l in STR_LOCI if l in df.columns])}/{len(STR_LOCI)} loci"
    )]
    _state["df"]            = df
    _state["file_name"]     = "sample_dataset_builtin.xlsx"
    _state["cancel"]        = False
    _state["wiped"]     = False
    _state["sheet_summary"] = sample_summary
    show_preview(df, "Built-in Sample Dataset",
                 "sample_dataset_builtin.xlsx",
                 sheet_summary=sample_summary)
    with out_status:
        display(_html(card(
            f"✅ <b>Sample data loaded.</b> &nbsp;"
            f"<span style='font-size:12px;color:{C['muted']}'>"
            f"Set mismatch threshold then click <b>Run Analysis</b>.</span>",
            bc=C["teal"], bg="#f0f7f5"
        )))

btn_sample.on_click(on_load_sample)

def on_upload(change):
    import pandas as pd
    if not uploader.value: return
    if len(uploader.value) > 1:
        _reset_uploader()
        with out_status:
            display(_html(card(
                "<b>Please select only one file at a time.</b> "
                "The upload has been cleared - please try again.",
                bc=C['red'], bg='#fff5f5'
            )))
        return
    fname, fdata = list(uploader.value.items())[0]
    content  = fdata["content"] if isinstance(fdata,dict) else bytes(fdata)
    fsize_kb = len(content) / 1024
    for o in [out_file,out_preview,out_progress,out_results,
              out_download,out_status,out_wipe_conf]:
        o.clear_output()
    try:
        sheet_summary = None
        if fname.lower().endswith(".csv"):
            df = pd.read_csv(io.BytesIO(content))
            kit_info = detect_kit(df.columns.tolist())
            sheet_summary = [dict(
                sheet_name="CSV",
                n_profiles=len(df),
                loci_count=len([l for l in STR_LOCI if l in df.columns]),
                loci_found=[l for l in STR_LOCI if l in df.columns],
                kit=kit_info["name"],
                kit_manufacturer=kit_info["manufacturer"],
                kit_notes=kit_info["notes"],
                absent_core=kit_info["absent_core"],
                status=f"✅ {len(df)} profiles"
            )]
        else:
            xl = pd.ExcelFile(io.BytesIO(content))
            if len(xl.sheet_names) >= 1:
                df, sheet_summary, conflict_warns = load_excel_sheets(content)
                if conflict_warns:
                    _state["conflict_warnings"] = conflict_warns
            else:
                df = xl.parse(xl.sheet_names[0])

        _state["df"]            = df
        _state["file_name"]     = fname
        _state["cancel"]        = False
        _state["wiped"]        = False
        _state["sheet_summary"] = sheet_summary
        show_preview(df, "User-uploaded file", fname, fsize_kb, sheet_summary)

        n_sheets = len(sheet_summary) if sheet_summary else 1
        used     = len([s for s in (sheet_summary or []) if s["n_profiles"] > 0])
        msg = (
            f"✅ <b>{fname}</b> uploaded — "
            f"<b>{n_sheets}</b> sheet{'s' if n_sheets>1 else ''} detected, "
            f"<b>{used}</b> with valid STR data, "
            f"<b>{len(df)}</b> profiles total.<br>"
            f"<span style='font-size:12px;color:{C['muted']}'>"
            f"Set mismatch threshold then click <b>Run Analysis</b>.</span>"
        )
        with out_status:
            display(_html(card(msg, bc=C["teal"], bg="#f0f7f5")))

        # Surface PowerPlex 16 warning immediately on upload
        for s in (sheet_summary or []):
            if s.get("absent_core"):
                with out_status:
                    display(_html(card(
                        f"⚠ <b>{s['sheet_name']} — {s['kit']} detected:</b> "
                        f"{s['kit_notes']}",
                        bc=C["red"], bg="#fff5f5"
                    )))

    except Exception as e:
        with out_status:
            display(_html(card(
                f"❌ <b>Could not read file:</b> {e}",
                bc=C["red"], bg="#fff5f5"
            )))

uploader.observe(on_upload, names="value")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — RUN ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

def on_run(_):
    for o in [out_progress,out_results,out_download,out_status,out_wipe_conf]:
        o.clear_output()
    # Scroll the tool header back into view (Colab-compatible)
    display(HTML(
        "<script>(function(){"
        "var el=document.getElementById('pykinshipid-top');"
        "if(el){el.scrollIntoView({behavior:'smooth',block:'start'});}"
        "})();</script>"
    ))

    # Block re-analysis after a wipe — uploader may still hold bytes
    if _state.get("wiped"):
        with out_status:
            display(_html(card(
                f"⚠ <b>Session was wiped.</b> Please upload a new file "
                f"or click <b>Load Sample Data</b> before running analysis.",
                bc=C["amber"], bg="#fffbf0"
            )))
        return

    if "df" not in _state and uploader.value:
        on_upload(None)

    if "df" not in _state:
        with out_status:
            display(_html(card(
                f"⚠ <b>No data loaded.</b> Upload a file or click "
                f"<b>Load Sample Data</b> first.",
                bc=C["amber"], bg="#fffbf0"
            )))
        return

    n = len(_state["df"])
    if n > 500:
        with out_status:
            display(_html(card(
                f"ℹ <b>Large dataset — {n} profiles.</b> "
                f"Estimated time: <b>{estimate_time(n)}</b>. "
                f"Keep this tab active.",
                bc=C["amber"], bg="#fffbf0"
            )))

    for w in _state.pop("conflict_warnings", []):
        with out_status:
            display(_html(card(f"⚠ {w}", bc=C["red"], bg="#fff5f5")))

    _state["cancel"] = False
    btn_cancel.layout.display = ""
    btn_analyse.layout.display = "none"
    t0 = time.time()

    def progress_cb(phase, pct, elapsed):
        out_progress.clear_output(wait=True)
        with out_progress:
            display(_html(progress_bar_html(phase, pct, elapsed)))

    def cancel_check():
        return _state.get("cancel", False)

    try:
        results    = run_analysis(
            _state["df"].copy(), slider_mm.value,
            progress_cb, cancel_check
        )
        elapsed    = time.time() - t0
        is_partial = results.get("cancelled", False)

        if results.get("column_renames") or results.get("col_warnings"):
            msgs = results.get("col_warnings",[])
            if results.get("column_renames"):
                msgs.append("Column names auto-corrected: " +
                            ", ".join(results["column_renames"]))
            with out_status:
                display(_html(card("ℹ " + "<br>".join(msgs), bc=C["slate"])))

        summary = (
            f"{len(results['profiles'])} profiles · "
            f"{len(results['trios'])} Trios · "
            f"{len(results['duos'])} Duos · "
            f"{len(results['duplicates'])} Dup sets · "
            f"{len(results['mixtures'])} Mixtures · "
            f"{len(results['individuals'])} Unrelated · "
            f"{elapsed:.1f}s"
            + (" [PARTIAL]" if is_partial else "")
        )
        w_header.value = make_header(summary)

        sheet_summary = _state.get("sheet_summary")
        report_html = build_report(
            results, _state.get("file_name","unknown"),
            elapsed, slider_mm.value, is_partial,
            sheet_summary=sheet_summary
        )
        _state["report_html"] = report_html

        out_progress.clear_output()
        with out_results:
            display(HTML(report_html))

        b64 = base64.b64encode(report_html.encode()).decode()
        ts  = datetime.now().strftime("%Y%m%d_%H%M%S")
        fn  = f"PyKinshipID_Report_{ts}.html"
        with out_download:
            display(_html(card(
                f"<b style='color:{C['navy']}'>📥 Report Ready</b>"
                f"<div style='margin-top:10px'>"
                f"<a href='data:text/html;base64,{b64}' download='{fn}' "
                f"style='padding:10px 22px;background:{C['navy']};color:#fff;"
                f"text-decoration:none;border-radius:4px;font-weight:bold;"
                f"font-size:13px;display:inline-block'>💾 Download Report (HTML)</a>"
                f"</div>"
                f"<div style='font-size:11px;color:{C['muted']};margin-top:10px'>"
                f"Open the downloaded file → click <b>Print / Save PDF</b> in the "
                f"report header, or press <b>Ctrl+P</b> / <b>Cmd+Shift+P</b> (Mac)."
                f"<br>⚠ When done, click <b>Wipe Session</b> below to clear all data.</div>",
                bc=C["teal"], bg="#f0f7f5"
            )))
            display(widgets.HBox([btn_wipe2]))

    except Exception as e:
        import traceback
        out_progress.clear_output()
        with out_results:
            display(_html(card(
                f"❌ <b>Analysis error:</b> {e}<br>"
                f"<pre style='font-size:11px;margin-top:6px;white-space:pre-wrap'>"
                f"{traceback.format_exc()}</pre>",
                bc=C["red"], bg="#fff5f5"
            )))
    finally:
        btn_cancel.layout.display = "none"
        btn_analyse.layout.display = ""

def on_cancel(_):
    _state["cancel"] = True
    with out_progress:
        display(_html(card(
            f"⛔ <b>Cancellation requested.</b> "
            f"Stopping after current phase completes…",
            bc=C["red"], bg="#fff5f5"
        )))

btn_analyse.on_click(on_run)
btn_cancel.on_click(on_cancel)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 — RENDER
# ─────────────────────────────────────────────────────────────────────────────

upload_row = widgets.HBox([uploader, btn_sample])

display(_html(
    f"<div style='background:{C['navy']};color:#fff;"
    f"padding:12px 20px;border-radius:4px;font-size:14px;font-weight:bold'>"
    f"🧬 Initialising {APP_TITLE}…</div>"
))

ui = widgets.VBox([
    w_header,
    w_steps,
    _html(_FORMAT_HTML),
    divider,
    _html(_SAMPLE_HTML),
    _html(sec_label("Load Data",
          "Upload your file or use the built-in sample dataset")),
    upload_row,
    _html(f"<div style='font-size:11px;color:{C['muted']};margin-bottom:4px'>"
          f"Upload <b>one</b> .xlsx (single or multi-sheet) or .csv file. "
          f"Kit auto-detected per sheet. Column names normalised automatically.<br>"
          f"<span style='color:{C['slate']}'>The number in brackets on the upload "
          f"button shows files in the buffer - it should show (1) after a "
          f"successful upload.</span></div>"),
    out_file, out_preview,
    divider,
    _html(sec_label("Analysis Settings",
          "Applies to both Trio and Duo screening")),
    slider_mm,
    _html(f"<div style='font-size:11px;color:{C['muted']};margin:2px 0 10px'>"
          f"0 = strict Mendelian match &nbsp;·&nbsp; "
          f"1–2 = allows allelic dropout / mutation &nbsp;·&nbsp; "
          f"Standard DVI practice: 1 mismatch</div>"),
    widgets.HBox([btn_analyse, btn_cancel]),
    out_status, out_progress, out_results, out_download,
    out_wipe_conf,
    divider,
    _html(sec_label("Session Privacy", "Clear all data when finished")),
    _html(card(
        f"<span style='font-size:12px;color:{C['text']}'>"
        f"Click <b>Wipe Session</b> to clear all loaded data, results, and "
        f"uploaded files. For complete assurance use "
        f"<b>Runtime → Restart Runtime</b> in the Colab menu. "
        f"Each user who opens this link receives a fresh isolated environment.</span>",
        bc=C["border"]
    )),
    widgets.HBox([btn_wipe1]),
    out_wipe_conf,
    _html(
        f"<div style='font-size:10px;color:{C['muted']};text-align:center;"
        f"padding:8px 0;border-top:1px solid {C['border']};margin-top:6px'>"
        f"{VERSION} &nbsp;·&nbsp; For research and screening use only &nbsp;·&nbsp; "
        f"Confirm all findings with LR-based forensic analysis</div>"
    )
])

clear_output(wait=True)
display(ui)